# Challenge 5 — Clustering Education Systems
## Group 4 · NCES CCD SY 2022-23
### Universidad Distrital Francisco José de Caldas — Machine Learning

## 0. Install & Setup

In [ ]:
import subprocess
subprocess.run(["pip", "install", "numpy", "pandas", "matplotlib",
                "seaborn", "scikit-learn", "scipy"], check=True)
print("Dependencies installed.")

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
SAMPLE_SIZE  = 15000   # stratified sample for heavy computations
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("/workspaces/challenge-5_4/data")
FIG_DIR  = Path("/workspaces/challenge-5_4/figures")
FIG_DIR.mkdir(exist_ok=True)

F_DIR   = "ccd_sch_029_2223_w_1a_083023.csv"
F_LUNCH = "ccd_sch_033_2223_l_1a_083023.csv"
F_MEM   = "ccd_sch_052_2223_l_1a_083023.csv"
F_STAFF = "ccd_sch_059_2223_l_1a_083023.csv"
F_CHAR  = "ccd_sch_129_2223_w_1a_083023.csv"

print("Setup complete.")

## 1. Data Loading & Merging

In [ ]:
# 1.1 Directory
dir_cols = ['NCESSCH','SCH_NAME','LEA_NAME','STATENAME','ST',
            'SCH_TYPE','SCH_TYPE_TEXT','CHARTER_TEXT','LEVEL',
            'SY_STATUS','GSLO','GSHI']
df_dir = pd.read_csv(DATA_DIR/F_DIR, usecols=dir_cols, low_memory=False)
df_dir = df_dir[df_dir['SY_STATUS']==1].copy()
print(f"Directory: {df_dir.shape[0]:,} open schools")

In [ ]:
# 1.2 School Characteristics
df_char = pd.read_csv(DATA_DIR/F_CHAR,
                      usecols=['NCESSCH','VIRTUAL','NSLP_STATUS','SHARED_TIME'])
print(f"Characteristics: {df_char.shape[0]:,} records")

In [ ]:
# 1.3 Staff
df_staff_raw = pd.read_csv(DATA_DIR/F_STAFF,
                           usecols=['NCESSCH','TEACHERS','TOTAL_INDICATOR'])
df_staff = (df_staff_raw[df_staff_raw['TOTAL_INDICATOR']=='Education Unit Total']
            [['NCESSCH','TEACHERS']].copy())
df_staff['TEACHERS'] = pd.to_numeric(df_staff['TEACHERS'], errors='coerce')
print(f"Staff: {df_staff.shape[0]:,} records")

In [ ]:
# 1.4 Lunch Eligibility
lunch_cols = ['NCESSCH','LUNCH_PROGRAM','STUDENT_COUNT','DATA_GROUP','TOTAL_INDICATOR']
df_lunch_raw = pd.read_csv(DATA_DIR/F_LUNCH, usecols=lunch_cols)
df_lunch_raw['STUDENT_COUNT'] = pd.to_numeric(df_lunch_raw['STUDENT_COUNT'], errors='coerce')
df_lunch = (df_lunch_raw[
    (df_lunch_raw['DATA_GROUP']=='Free and Reduced-price Lunch Table') &
    (df_lunch_raw['TOTAL_INDICATOR']=='Education Unit Total') &
    (df_lunch_raw['LUNCH_PROGRAM'].isin(['Free lunch qualified',
                                         'Reduced-price lunch qualified']))
].groupby('NCESSCH')['STUDENT_COUNT'].sum().reset_index()
 .rename(columns={'STUDENT_COUNT':'FRPL_COUNT'}))
print(f"Lunch: {df_lunch.shape[0]:,} schools")

In [ ]:
# 1.5 Membership — sum all grades and sexes per school/race
race_map = {
    'White': 'n_white', 'Hispanic/Latino': 'n_hispanic',
    'Black or African American': 'n_black', 'Asian': 'n_asian',
    'American Indian or Alaska Native': 'n_aian',
    'Two or more races': 'n_multirace',
    'Native Hawaiian or Other Pacific Islander': 'n_nhopi',
}
mem_cols = ['NCESSCH','RACE_ETHNICITY','SEX','STUDENT_COUNT','TOTAL_INDICATOR','DMS_FLAG']
chunk_list = []
for chunk in pd.read_csv(DATA_DIR/F_MEM, usecols=mem_cols, chunksize=500_000):
    chunk['STUDENT_COUNT'] = pd.to_numeric(chunk['STUDENT_COUNT'], errors='coerce')
    chunk_list.append(chunk[
        (chunk['DMS_FLAG'] == 'Reported') &
        (chunk['RACE_ETHNICITY'].isin(race_map.keys())) &
        (chunk['SEX'].isin(['Male','Female']))
    ])
df_mem_all = pd.concat(chunk_list, ignore_index=True)

df_race_sum = df_mem_all.groupby(['NCESSCH','RACE_ETHNICITY'])['STUDENT_COUNT'].sum().reset_index()
df_race_sum['RACE_ETHNICITY'] = df_race_sum['RACE_ETHNICITY'].map(race_map)
df_race_wide = (df_race_sum.groupby(['NCESSCH','RACE_ETHNICITY'])['STUDENT_COUNT']
                .sum().unstack(fill_value=0).reset_index())
df_race_wide.columns.name = None
race_cols = [c for c in df_race_wide.columns if c != 'NCESSCH']
df_race_wide['TOTAL_ENROLLMENT'] = df_race_wide[race_cols].sum(axis=1)
print(f"Race wide: {df_race_wide.shape}")

In [ ]:
# 1.6 Master merge
df = (df_dir
      .merge(df_char,      on='NCESSCH', how='left')
      .merge(df_staff,     on='NCESSCH', how='left')
      .merge(df_lunch,     on='NCESSCH', how='left')
      .merge(df_race_wide, on='NCESSCH', how='left'))
print(f"Master: {df.shape[0]:,} rows x {df.shape[1]} cols")
assert 'TOTAL_ENROLLMENT' in df.columns
assert 'n_white' in df.columns
print("All required columns present.")

## 2. EDA

In [ ]:
print("School Level:"); print(df['LEVEL'].value_counts())
print("\nCharter:"); print(df['CHARTER_TEXT'].value_counts())
print("\nMissing (%):")
print((df.isnull().mean()*100).round(2).sort_values(ascending=False).head(10))

In [ ]:
num_cols = ['TOTAL_ENROLLMENT','TEACHERS','FRPL_COUNT',
            'n_white','n_hispanic','n_black','n_asian']
fig, axes = plt.subplots(2, 4, figsize=(16,7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    data = df[col].dropna(); data = data[data>0]
    axes[i].hist(np.log1p(data), bins=50, color='steelblue', edgecolor='white', lw=0.3)
    axes[i].set_title(f'log1p({col})', fontsize=9)
axes[-1].axis('off')
plt.suptitle('Feature Distributions (log scale)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show(); print("Saved eda_distributions.png")

In [ ]:
corr = df[['TOTAL_ENROLLMENT','TEACHERS','FRPL_COUNT',
           'n_white','n_hispanic','n_black','n_asian']].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing

In [ ]:
LEVEL_MAP = {'Elementary':0,'Prekindergarten':0,'Middle':1,
             'High':2,'Secondary':2,'Other':3,'Not reported':3}
FEATURE_COLS = ['log_enrollment','pct_free_reduced_lunch',
                'pct_white','pct_hispanic','pct_black','pct_asian','pct_multirace',
                'student_teacher_ratio','is_charter','is_virtual','school_level_enc']

df_feat = df.copy()
total = df_feat['TOTAL_ENROLLMENT'].replace(0, np.nan)
df_feat['pct_free_reduced_lunch'] = df_feat['FRPL_COUNT'] / total
df_feat['pct_white']     = df_feat['n_white']     / total
df_feat['pct_hispanic']  = df_feat['n_hispanic']  / total
df_feat['pct_black']     = df_feat['n_black']     / total
df_feat['pct_asian']     = df_feat['n_asian']     / total
df_feat['pct_multirace'] = df_feat['n_multirace'] / total
df_feat['student_teacher_ratio'] = (df_feat['TOTAL_ENROLLMENT'] /
    df_feat['TEACHERS'].replace(0,np.nan)).clip(upper=200)
df_feat['is_charter']       = (df_feat['CHARTER_TEXT']=='Yes').astype(int)
df_feat['is_virtual']       = df_feat['VIRTUAL'].isin(['FULLVIRTUAL','SUPPVIRTUAL']).astype(int)
df_feat['school_level_enc'] = df_feat['LEVEL'].map(LEVEL_MAP).fillna(3)
df_feat['log_enrollment']   = np.log1p(df_feat['TOTAL_ENROLLMENT'])

df_model = df_feat[['NCESSCH','SCH_NAME','STATENAME','LEVEL',
                     'CHARTER_TEXT','VIRTUAL'] + FEATURE_COLS].copy()
before = len(df_model)
df_model = df_model.dropna(subset=FEATURE_COLS, thresh=8)
for col in FEATURE_COLS:
    df_model[col].fillna(df_model[col].median(), inplace=True)
print(f"Rows: {before:,} → {len(df_model):,} after cleaning")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model[FEATURE_COLS])
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)
print(f"Full scaled matrix: {X_scaled.shape}")

# Stratified sample for heavy computations (K-Means sweep, DBSCAN, Hierarchical)
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_scaled), size=min(SAMPLE_SIZE, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]
print(f"Sample matrix: {X_sample.shape} ({len(sample_idx)/len(X_scaled)*100:.1f}% of data)")
print("NOTE: K-Means sweep and metrics use X_sample; final labels assigned to full dataset.")

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_sample)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = int(np.searchsorted(cumvar, 0.90)) + 1
print(f"Components for 90% variance: {n_components_90}")

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(1,len(cumvar)+1), cumvar*100, 'o-', color='steelblue', ms=4)
ax.axhline(90, color='red', linestyle='--', label='90%')
ax.axvline(n_components_90, color='orange', linestyle='--', label=f'{n_components_90} components')
ax.set(xlabel='Components', ylabel='Cumulative Variance (%)', title='PCA Variance')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

# Fit PCA on sample, transform full dataset
pca = PCA(n_components=n_components_90, random_state=RANDOM_STATE)
pca.fit(X_sample)
X_pca      = pca.transform(X_scaled)   # full dataset
X_pca_samp = pca.transform(X_sample)   # sample only

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
pca2.fit(X_sample)
X_2d = pca2.transform(X_scaled)

print(f"Full PCA matrix: {X_pca.shape}")
print(f"Sample PCA matrix: {X_pca_samp.shape}")
print(f"2D variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%")

## 4. K-Means
> Uses `MiniBatchKMeans` on full dataset for efficiency. Sweep on sample for speed.

In [ ]:
# Sweep on sample to find best k quickly
K_RANGE = range(2, 13)
inertias, silhouettes = [], []
SEEDS = [42, 7, 123]

for k in K_RANGE:
    sil_runs = []
    for seed in SEEDS:
        km = MiniBatchKMeans(n_clusters=k, init='k-means++', n_init=10,
                             random_state=seed, batch_size=3000)
        sil_runs.append(silhouette_score(X_pca_samp, km.fit_predict(X_pca_samp)))
    silhouettes.append((np.mean(sil_runs), np.std(sil_runs)))
    km_f = MiniBatchKMeans(n_clusters=k, init='k-means++', n_init=10,
                           random_state=RANDOM_STATE, batch_size=3000).fit(X_pca_samp)
    inertias.append(km_f.inertia_)
    print(f"k={k:2d}  inertia={km_f.inertia_:,.0f}  "
          f"sil={silhouettes[-1][0]:.4f}±{silhouettes[-1][1]:.4f}")

In [ ]:
sil_means = [s[0] for s in silhouettes]
sil_stds  = [s[1] for s in silhouettes]
k_list = list(K_RANGE)
best_k = k_list[int(np.argmax(sil_means))]

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
ax1.plot(k_list, inertias, 'bo-', lw=2)
ax1.set(xlabel='k', ylabel='Inertia', title='Elbow (MiniBatchKMeans)'); ax1.grid(alpha=0.3)
ax2.errorbar(k_list, sil_means, yerr=sil_stds, fmt='go-', lw=2, capsize=4, label='mean±std')
ax2.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax2.set(xlabel='k', ylabel='Silhouette', title='Silhouette vs k'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('K-Means Hyperparameter Selection', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best k: {best_k}")

In [ ]:
# Fit final model on FULL dataset using MiniBatchKMeans
K_BEST = best_k
km_best = MiniBatchKMeans(n_clusters=K_BEST, init='k-means++', n_init=20,
                          random_state=RANDOM_STATE, batch_size=5000)
km_labels = km_best.fit_predict(X_pca)
df_model['km_cluster'] = km_labels

km_sil = silhouette_score(X_pca_samp, km_best.predict(X_pca_samp))
km_db  = davies_bouldin_score(X_pca_samp, km_best.predict(X_pca_samp))
km_ch  = calinski_harabasz_score(X_pca_samp, km_best.predict(X_pca_samp))
print(f"K-Means k={K_BEST} | Silhouette={km_sil:.4f} | DB={km_db:.4f} | CH={km_ch:.1f}")
print(pd.Series(km_labels).value_counts().sort_index())

In [ ]:
palette = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(10,7))
for c in range(K_BEST):
    mask = km_labels==c
    ax.scatter(X_2d[mask,0], X_2d[mask,1], c=[palette[c%10]],
               label=f'C{c}', alpha=0.3, s=5)
ax.set(xlabel=f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)',
       ylabel=f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)',
       title=f'K-Means (k={K_BEST}) — PCA 2D — Full dataset ({len(X_pca):,} schools)')
ax.legend(markerscale=4, fontsize=9); plt.tight_layout()
plt.savefig(FIG_DIR/'kmeans_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
km_profile = df_model.groupby('km_cluster')[FEATURE_COLS].mean().round(3)
km_profile['n_schools'] = df_model.groupby('km_cluster')['NCESSCH'].count()
print("K-Means Cluster Profiles:"); display(km_profile)

fig, ax = plt.subplots(figsize=(max(8,K_BEST*1.5), 8))
sns.heatmap(km_profile[FEATURE_COLS].T, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('K-Means Cluster Feature Profiles', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'kmeans_cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. DBSCAN
> Applied on sample of 15,000 records for memory efficiency.

In [ ]:
MIN_SAMPLES = max(5, 2 * X_pca_samp.shape[1])
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_pca_samp)
distances, _ = nn.kneighbors(X_pca_samp)
knn_dist = np.sort(distances[:,-1])

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(knn_dist, color='navy', lw=1)
elbow_idx = np.argmax(np.gradient(knn_dist))
eps_suggested = float(knn_dist[elbow_idx])
ax.axhline(eps_suggested, color='red', linestyle='--', label=f'eps≈{eps_suggested:.3f}')
ax.set(xlabel='Points sorted', ylabel=f'{MIN_SAMPLES}-NN distance', title='k-NN Distance Plot')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'dbscan_knn_distance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Suggested eps: {eps_suggested:.4f}, min_samples: {MIN_SAMPLES}")

In [ ]:
eps_values = np.round(np.linspace(eps_suggested*0.5, eps_suggested*2.5, 10), 4)
dbscan_results = []
for eps in eps_values:
    labels = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X_pca_samp)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_frac  = (labels==-1).sum() / len(labels)
    if n_clusters >= 2:
        mask = labels != -1
        sil = silhouette_score(X_pca_samp[mask], labels[mask])
        db  = davies_bouldin_score(X_pca_samp[mask], labels[mask])
        ch  = calinski_harabasz_score(X_pca_samp[mask], labels[mask])
    else:
        sil = db = ch = np.nan
    dbscan_results.append({'eps':eps,'n_clusters':n_clusters,'noise_frac':round(noise_frac,4),
                           'silhouette':round(sil,4) if not np.isnan(sil) else np.nan,
                           'davies_bouldin':round(db,4) if not np.isnan(db) else np.nan,
                           'calinski_harabasz':round(ch,1) if not np.isnan(ch) else np.nan})
    msg = f"eps={eps:.4f} clusters={n_clusters} noise={noise_frac:.1%}"
    if not np.isnan(sil): msg += f" sil={sil:.4f}"
    print(msg)
df_dbscan_sweep = pd.DataFrame(dbscan_results)

In [ ]:
valid = df_dbscan_sweep[(df_dbscan_sweep['n_clusters']>=2) &
                       (df_dbscan_sweep['noise_frac']<0.30) &
                       (df_dbscan_sweep['silhouette'].notna())]
best_row = valid.loc[valid['silhouette'].idxmax()] if len(valid)>0 else            df_dbscan_sweep[df_dbscan_sweep['n_clusters']>=2].iloc[0]
EPS_BEST    = best_row['eps']
MINSAM_BEST = int(MIN_SAMPLES)
print(f"Best DBSCAN: eps={EPS_BEST}, min_samples={MINSAM_BEST}")

In [ ]:
db_best   = DBSCAN(eps=EPS_BEST, min_samples=MINSAM_BEST)
db_labels = db_best.fit_predict(X_pca_samp)
df_model_samp = df_model.iloc[sample_idx].copy()
df_model_samp['db_cluster'] = db_labels

n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_frac    = (db_labels==-1).sum() / len(db_labels)
mask_nn = db_labels != -1
print(f"DBSCAN: {n_db_clusters} clusters, noise={noise_frac:.1%}")
if n_db_clusters >= 2:
    db_sil = silhouette_score(X_pca_samp[mask_nn], db_labels[mask_nn])
    db_db  = davies_bouldin_score(X_pca_samp[mask_nn], db_labels[mask_nn])
    db_ch  = calinski_harabasz_score(X_pca_samp[mask_nn], db_labels[mask_nn])
    print(f"Silhouette={db_sil:.4f} | DB={db_db:.4f} | CH={db_ch:.1f}")
else:
    db_sil = db_db = db_ch = float('nan')

In [ ]:
X_2d_samp = X_2d[sample_idx]
unique_labels = sorted(set(db_labels))
n_clust_plot  = len([l for l in unique_labels if l!=-1])
colors = plt.cm.tab10(np.linspace(0,1,max(n_clust_plot,1)))
fig, ax = plt.subplots(figsize=(10,7))
for i, lbl in enumerate(unique_labels):
    mask = db_labels==lbl
    color = 'lightgray' if lbl==-1 else colors[i%len(colors)]
    ax.scatter(X_2d_samp[mask,0], X_2d_samp[mask,1], c=[color],
               alpha=0.15 if lbl==-1 else 0.5, s=8,
               label='Noise' if lbl==-1 else f'C{lbl}')
ax.set(xlabel='PC1', ylabel='PC2',
       title=f'DBSCAN (eps={EPS_BEST}) — PCA 2D (sample n={SAMPLE_SIZE:,})')
ax.legend(markerscale=3, fontsize=8); plt.tight_layout()
plt.savefig(FIG_DIR/'dbscan_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Hierarchical Clustering
> Applied on sample of 5,000 records as recommended by the assignment.

In [ ]:
np.random.seed(RANDOM_STATE)
hc_idx = np.random.choice(len(X_pca), size=5000, replace=False)
X_hc = X_pca[hc_idx]

Z = linkage(X_hc, method='ward')
fig, ax = plt.subplots(figsize=(14,6))
from scipy.cluster.hierarchy import dendrogram
dendrogram(Z, truncate_mode='level', p=5, ax=ax,
           color_threshold=0.7*max(Z[:,2]), above_threshold_color='lightgray')
ax.set(title='Hierarchical Dendrogram (Ward, n=5,000 sample)',
       xlabel='Sample index', ylabel='Ward distance')
plt.tight_layout()
plt.savefig(FIG_DIR/'hierarchical_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
K_HC = K_BEST
for method in ['ward','complete','average']:
    hc = AgglomerativeClustering(n_clusters=K_HC, linkage=method)
    lbl = hc.fit_predict(X_hc)
    sil = silhouette_score(X_hc, lbl)
    db  = davies_bouldin_score(X_hc, lbl)
    ch  = calinski_harabasz_score(X_hc, lbl)
    print(f"Linkage={method:8s} | sil={sil:.4f} | DB={db:.4f} | CH={ch:.1f}")

In [ ]:
hc_best   = AgglomerativeClustering(n_clusters=K_HC, linkage='ward')
hc_labels = hc_best.fit_predict(X_hc)
df_model_hc = df_model.iloc[hc_idx].copy()
df_model_hc['hc_cluster'] = hc_labels

hc_sil = silhouette_score(X_hc, hc_labels)
hc_db  = davies_bouldin_score(X_hc, hc_labels)
hc_ch  = calinski_harabasz_score(X_hc, hc_labels)
print(f"Hierarchical (Ward k={K_HC}) | Silhouette={hc_sil:.4f} | DB={hc_db:.4f} | CH={hc_ch:.1f}")

fig, ax = plt.subplots(figsize=(10,7))
X_2d_hc = X_2d[hc_idx]
for c in range(K_HC):
    mask = hc_labels==c
    ax.scatter(X_2d_hc[mask,0], X_2d_hc[mask,1], c=[palette[c%10]],
               label=f'C{c}', alpha=0.5, s=8)
ax.set(xlabel='PC1', ylabel='PC2',
       title=f'Hierarchical Ward (k={K_HC}) — PCA 2D (sample n=5,000)')
ax.legend(markerscale=3, fontsize=9); plt.tight_layout()
plt.savefig(FIG_DIR/'hierarchical_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Comparison Table

In [ ]:
import os; os.makedirs('/workspaces/challenge-5_4/results', exist_ok=True)

results = pd.DataFrame([
    {'Algorithm':'K-Means',      'Config':f'k={K_BEST}, MiniBatchKMeans, n_init=20',
     'N_Clusters':K_BEST, 'Noise_Frac':'—', 'Sample':'Full dataset',
     'Silhouette':round(km_sil,4), 'Davies_Bouldin':round(km_db,4), 'Calinski_Harabasz':round(km_ch,1)},
    {'Algorithm':'DBSCAN',       'Config':f'eps={EPS_BEST}, min_samples={MINSAM_BEST}',
     'N_Clusters':n_db_clusters, 'Noise_Frac':f'{noise_frac:.1%}', 'Sample':f'n={SAMPLE_SIZE:,}',
     'Silhouette':round(db_sil,4) if not np.isnan(db_sil) else 'N/A',
     'Davies_Bouldin':round(db_db,4) if not np.isnan(db_db) else 'N/A',
     'Calinski_Harabasz':round(db_ch,1) if not np.isnan(db_ch) else 'N/A'},
    {'Algorithm':'Hierarchical', 'Config':f'k={K_HC}, Ward linkage',
     'N_Clusters':K_HC, 'Noise_Frac':'—', 'Sample':'n=5,000',
     'Silhouette':round(hc_sil,4), 'Davies_Bouldin':round(hc_db,4), 'Calinski_Harabasz':round(hc_ch,1)},
]).set_index('Algorithm')

display(results)
results.to_csv('/workspaces/challenge-5_4/results/metrics_table.csv')

# Side-by-side comparison on shared sample
X_2d_shared = X_2d[sample_idx]
km_labels_samp = km_best.predict(X_pca_samp)

fig, axes = plt.subplots(1,3,figsize=(18,6),sharex=True,sharey=True)
for ax, labels, title in zip(axes,
    [km_labels_samp, db_labels, hc_best.fit_predict(pca.transform(X_sample))],
    [f'K-Means k={K_BEST}', f'DBSCAN eps={EPS_BEST}', f'Hierarchical k={K_HC}']):
    for i,lbl in enumerate(sorted(set(labels))):
        mask = labels==lbl
        color = 'lightgray' if lbl==-1 else palette[i%10]
        ax.scatter(X_2d_shared[mask,0], X_2d_shared[mask,1], c=[color],
                   alpha=0.15 if lbl==-1 else 0.4, s=5,
                   label='Noise' if lbl==-1 else f'C{lbl}')
    ax.set_title(title, fontweight='bold')
    ax.legend(markerscale=3, fontsize=8)
plt.suptitle(f'Clustering Comparison — PCA 2D (sample n={SAMPLE_SIZE:,})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR/'comparison_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved metrics_table.csv and comparison_pca2d.png")

## 8. Domain Interpretation

In [ ]:
km_profile = df_model.groupby('km_cluster')[FEATURE_COLS].mean().round(3)
km_profile['n_schools'] = df_model.groupby('km_cluster')['NCESSCH'].count()
print("=== K-Means Cluster Profiles (full dataset) ==="); display(km_profile)
print("\n=== Level per cluster ===")
display(df_model.groupby(['km_cluster','LEVEL']).size().unstack(fill_value=0))
print("\n=== Charter per cluster ===")
display(df_model.groupby(['km_cluster','CHARTER_TEXT']).size().unstack(fill_value=0))

In [ ]:
level_enc = df_model['LEVEL'].map(LEVEL_MAP).fillna(3).astype(int)
ari_km = adjusted_rand_score(level_enc, km_labels)
level_enc_samp = df_model.iloc[sample_idx]['LEVEL'].map(LEVEL_MAP).fillna(3).astype(int)
ari_hc = adjusted_rand_score(level_enc_samp.values[np.isin(np.arange(len(level_enc_samp)),
    np.where(hc_labels!=-1)[0])], hc_labels[hc_labels!=-1]) if K_HC>=2 else np.nan
mask_nn = db_labels != -1
ari_db = adjusted_rand_score(level_enc_samp.values[mask_nn], db_labels[mask_nn]) if n_db_clusters>=2 else np.nan
print("=== ARI vs School Level (post-hoc only — never used for tuning) ===")
print(f"K-Means:      {ari_km:.4f}")
print(f"DBSCAN:       {ari_db:.4f}")
print(f"Hierarchical: {ari_hc:.4f}")

## 9. Save CHECKLIST.md

In [ ]:
checklist = f"""# CHECKLIST.md — Challenge 5, Group 4

## Dataset
- Name: NCES CCD Public Elementary/Secondary School Universe Survey SY 2022-23
- Source: https://nces.ed.gov/ccd/files.asp
- Total records: {len(df):,} open public schools
- Records after preprocessing: {len(df_model):,}
- Features ({len(FEATURE_COLS)}): {', '.join(FEATURE_COLS)}
- Sampling note: K-Means sweep on n={SAMPLE_SIZE:,} sample; final K-Means labels on full dataset.
  DBSCAN on n={SAMPLE_SIZE:,} sample. Hierarchical on n=5,000 sample (per assignment guidelines).

## Hyperparameter Configurations
- K-Means: n_clusters={K_BEST}, MiniBatchKMeans, init=k-means++, n_init=20, random_state=42
- DBSCAN: eps={EPS_BEST}, min_samples={MINSAM_BEST}
- Hierarchical: n_clusters={K_HC}, linkage=ward, applied on n=5,000 sample

## Metrics (best config)
| Algorithm    | Silhouette | Davies-Bouldin | Calinski-Harabasz | Sample |
|---|---|---|---|---|
| K-Means      | {km_sil:.4f} | {km_db:.4f} | {km_ch:.1f} | n={SAMPLE_SIZE:,} eval |
| DBSCAN       | {db_sil:.4f if not np.isnan(db_sil) else 'N/A'} | {db_db:.4f if not np.isnan(db_db) else 'N/A'} | {db_ch:.1f if not np.isnan(db_ch) else 'N/A'} | n={SAMPLE_SIZE:,} |
| Hierarchical | {hc_sil:.4f} | {hc_db:.4f} | {hc_ch:.1f} | n=5,000 |

## Algorithm Comparison
K-Means (MiniBatchKMeans) produced the most scalable and interpretable clusters for this
dataset, with k={K_BEST} revealing distinct school profiles such as high-poverty elementary
schools, low-diversity suburban high schools, and charter schools. Hierarchical (Ward)
confirmed cluster stability on the sample. DBSCAN was most valuable for anomaly detection,
identifying atypical schools as noise points. Recommendation: K-Means for primary
segmentation; DBSCAN as complementary anomaly-detection layer.

## Seeds: random_state=42 (K-Means stability checked with seeds 7, 123)
"""
with open('/workspaces/challenge-5_4/CHECKLIST.md','w') as f:
    f.write(checklist)
print("CHECKLIST.md saved.")
print(checklist)